In [1]:
import pandas as pd


In [2]:
roll_number = "1024160118"

In [3]:
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    },
]

In [4]:
last_two_digits = [int(d) for d in roll_number[-2:]]

In [5]:
categories = ["billing", "account", "general"]

In [6]:
personalized_entries = []

In [7]:
d = last_two_digits[0]
category = categories[d % 3]

personalized_entries.append({
    "question": "how do i update my registered mobile number",
    "answer": "Go to Account Settings and select Update Mobile Number.",
    "keywords": "mobile number update phone",
    "category": category
})

In [8]:
d = last_two_digits[1]
category = categories[d % 3]

personalized_entries.append({
    "question": "how can i contact customer support",
    "answer": "You can contact customer support through the help section.",
    "keywords": "support help contact service",
    "category": category
})

In [9]:
all_entries = fixed_entries + personalized_entries

df = pd.DataFrame(all_entries)

print("Q1: Final 6-row DataFrame")
print(df)

Q1: Final 6-row DataFrame
                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5           how can i contact customer support   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to Account Settings and select Update Mobil...   
5  You can contact customer support through the h...   

                       keywords category  
0         fee cost price charge  billing  
1          password reset login  account  
2        hours timing open time  general  
3           pay payment upi fee  billing  
4    mobile num

In [10]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())

        # Score = matching keywords + matching question words
        keyword_matches = query_words.intersection(keyword_words)
        question_matches = query_words.intersection(question_words)

        score = len(keyword_matches) + len(question_matches)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results

In [11]:
print("\nQ2: Scoring example")
query = "how do i reset my password"
results = score_query(query, df)

for result in results:
    print(result)


Q2: Scoring example
{'index': 1, 'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'category': 'account', 'score': 5}
{'index': 4, 'question': 'how do i update my registered mobile number', 'answer': 'Go to Account Settings and select Update Mobile Number.', 'category': 'account', 'score': 4}
{'index': 3, 'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 2}
{'index': 5, 'question': 'how can i contact customer support', 'answer': 'You can contact customer support through the help section.', 'category': 'general', 'score': 2}


In [12]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

# Category of first personalized entry = account
personalized_category = personalized_entries[0]["category"]

print("\nQ3: Entries in category:", personalized_category)
print(same_category(personalized_category, df))


Q3: Entries in category: account
                                      question  \
1                        how to reset password   
4  how do i update my registered mobile number   

                                              answer  \
1                   Go to Settings > Reset Password.   
4  Go to Account Settings and select Update Mobil...   

                     keywords category  
1        password reset login  account  
4  mobile number update phone  account  


In [13]:
print("\nQ4: Choose an entry to update")
print(df[["question", "keywords"]])

# Pick the first entry
entry_index = 0

new_keyword = input("\nEnter a new keyword to add: ").strip()

if new_keyword:
    df.loc[entry_index, "keywords"] += " " + new_keyword

# Save the complete updated DataFrame
filename = f"{roll_number}_faq_data.csv"
df.to_csv(filename, index=False)

print(f"Updated DataFrame saved to {filename}")
print(df)


Q4: Choose an entry to update
                                      question                      keywords
0                       what is the annual fee         fee cost price charge
1                        how to reset password          password reset login
2                  what are your working hours        hours timing open time
3                        how can i pay the fee           pay payment upi fee
4  how do i update my registered mobile number    mobile number update phone
5           how can i contact customer support  support help contact service

Enter a new keyword to add: as
Updated DataFrame saved to 1024160118_faq_data.csv
                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5           how can i contact customer support 

In [14]:
print("\nQ5: Number of FAQ entries per category")
category_counts = df.groupby("category").size()

print(category_counts)


Q5: Number of FAQ entries per category
category
account    2
billing    2
general    2
dtype: int64


In [15]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())

        keyword_matches = query_words.intersection(keyword_words)
        question_matches = query_words.intersection(question_words)

        score = len(keyword_matches) + len(question_matches)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if not results:
        print("No matching entries found.")
        return []

    results.sort(key=lambda x: x["score"], reverse=True)

    highest_score = results[0]["score"]
    highest_matches = [
        result for result in results
        if result["score"] == highest_score
    ]

    if len(highest_matches) > 1:
        print(f"\nTie found! {len(highest_matches)} entries have the highest score ({highest_score}):")
        for result in highest_matches:
            print(result)
    else:
        print("\nUnique highest-scoring match:")
        print(highest_matches[0])

    return results